# Snow Cover Extent analysis
J. Michelle Hu  
University of Utah  
June 2025  
---
Notebook comparing total basin snow cover extent against ASO reference (percent cover, difference in percent cover) by depth product [Baseline, HRRR-SPIReS, NWM, UA]
Calculations of early and delayed melt fractions compared to reference included.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import seaborn as sns

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

from pathlib import PurePath

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
diffdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'

basin = '*'
var = 'depth'
WY = '*'
drop_baseline = True

# Load the depth difference products
product_diff_fns = h.fn_list(diffdir, f'{basin}_wy{WY}*diff*{var}_original.nc')
product_diff = [xr.open_dataset(fn) for fn in product_diff_fns]

# Pull in ASO in the same grid
aso_grid_fns = h.fn_list(diffdir, f'{basin}_wy{WY}*aso*{var}_original.nc')
aso_grid = [xr.open_dataset(fn) for fn in aso_grid_fns]

# Compute the depth products from the product_diff and aso_grid
depth_products = [p['depth_diff']+a['aso_depth'] for p, a in zip(product_diff, aso_grid)]

In [ ]:
# Generate a uniform validity mask and apply to each product
# --> ASO has a couple more pixels in the NWM grid than the diff and depth product, need this step
for depth, aso, in zip(depth_products, aso_grid):
    # Create a mask for valid pixels
    valid_mask = ~np.isnan(depth) & ~np.isnan(aso['aso_depth'])
    # Apply the mask to both the depth and aso product
    depth.values[~valid_mask] = np.nan
    aso['aso_depth'].values[~valid_mask] = np.nan

# compute the valid pixels (each of these should be the same) into boolean grids
aso_valid_pixels = [~np.isnan(a['aso_depth']) for a in aso_grid]
product_valid_pixels = [~np.isnan(p['depth_diff']) for p in product_diff]
depth_products_valid_pixels = [~np.isnan(d) for d in depth_products]

# Sum valid pixels != 0 for each product --> this is the snow cover extent
aso_nonzero_pixels = [~np.isnan(a['aso_depth']) & (a['aso_depth'] > 0) for a in aso_grid]
product_nonzero_pixels = [~np.isnan(p['depth_diff']) & (p['depth_diff'] > 0) for p in product_diff]
depth_products_nonzero_pixels = [~np.isnan(d) & (d > 0) for d in depth_products]

for a, d, p in zip(aso_valid_pixels, depth_products_valid_pixels, product_valid_pixels):
    print(a.sum().values, d.sum().values, p.sum().values)

In [ ]:
# Compute snow cover extent and store the results in a dictionary for easy access
sce_results = {}
print('Ref SCE | Run SCE | Diff')
basind = ''
for a, d, p, t, f in zip(aso_nonzero_pixels, product_nonzero_pixels, depth_products_nonzero_pixels, aso_valid_pixels, product_diff_fns):
    basin = PurePath(f).stem.split('_')[0]
    dt_var = PurePath(f).stem.split('diff_')[1].split('_original')[0]
    runtype = PurePath(f).stem.split('_diff')[0].split('_')[-1]
    if basind != basin:
        print(basin)
    basind = basin

    total = t.sum().values
    ref_sce = a.sum().values / total * 100
    product_sce = p.sum().values / total * 100

    print(f'{ref_sce:.1f}% | {product_sce:.1f}% | {(product_sce - ref_sce):.1f}%')
    sce_results[f'{basin}_{dt_var}_{runtype}_aso'] = ref_sce / 100
    sce_results[f'{basin}_{dt_var}_{runtype}_depth'] = product_sce / 100

In [ ]:
# Convert dict to dataframe for plotting
df = pd.DataFrame(data=sce_results, index=[0]).T.rename(columns={0: 'SCE'})

# Add a column for the basin and the dt and the run types
df['basin'] = df.index.str.split('_').str[0]
df['dt'] = df.index.str.split('_').str[1]
df['runtype'] = df.index.str.split('_').str[3]
df['product'] = df.index.str.split('_').str[4]

# Append the product to the runtype with for each row if the product is 'aso'
# df.loc[df['product'] == 'aso', 'runtype'] = df['product']
df.loc[df['product'] == 'aso', 'runtype'] = df['runtype'] + '_aso'

# Drop the product column
df.drop(columns=['product'], inplace=True)

# And then reset the index
df.reset_index(inplace=True)

# And change the 'index' column name to filename, or drop it
df.rename(columns={'index': 'filename'}, inplace=True)

# Coerce dt to pandas datetime
df['dt'] = pd.to_datetime(df['dt'], format='%Y%m%d')

# Drop all rows that contain baseline in the filename
if drop_baseline:
    df = df[~df['filename'].str.contains('baseline')]
df

In [ ]:
# Create a separate df for each of the runtypes
sub_df_list = []
for r in np.unique(df['runtype']):
    sub_df = df[df['runtype'] == r].copy()
    # Append the runtype to the SCE column name
    sub_df.rename(columns={'SCE': f'SCE_{r}'}, inplace=True)
    # And append the runtype to the basin column name to avoid issues with concatentation
    sub_df.rename(columns={'basin': f'basin_{r}'}, inplace=True)
    # Reset the index
    sub_df.reset_index(drop=True, inplace=True)
    # Drop the filename column, drop the runtype column
    sub_df.drop(columns=['filename', 'runtype'], inplace=True)
    sub_df_list.append(sub_df)
sub_df_list[5]

In [ ]:
# Combine the sub_dfs together based on the dt column
df_final = pd.concat(sub_df_list, axis=1)
# Create a new basin column
df_final['basin'] = df_final['basin_hrrrspires']
# Drop the basin_ columns
df_final.drop(columns=[col for col in df_final.columns if col.startswith('basin_')], inplace=True)
# Remove all but the first dt column encountered
df_final = df_final.loc[:, ~df_final.columns.duplicated()]
# Set the dt column as the index
df_final.set_index('dt', inplace=True)
# Clean up the column names, set the dt column as the index
df_final

In [ ]:
df_diff

In [ ]:
for col in df_diff.columns:
    if 'basin' in col:
        continue
    else:
        _ = [print(f'{v*100:.0f}%') for v in df_diff[col] if not np.isnan(v)]

In [ ]:
df_final['SCE_nwm_aso'].index

In [ ]:
def plot_it_up(df_final, var='SCE', colors=None, barwidth=0.75, figsize=(16, 4), basiny=-0.2, barpadding=-25, ylims=None):
    fig, ax = plt.subplots(figsize=figsize)
    # Change the markers based on column names
    if colors is None:
        colors = {
                    f'{var}_hrrrspires_aso': 'lightblue',
                    f'{var}_nwm_aso': 'thistle',
                    f'{var}_ua_aso': 'burlywood',
                    f'{var}_nwm': 'purple',
                    f'{var}_hrrrspires': 'dodgerblue',
                    f'{var}_ua': 'gold',
                    f'{var}_baseline': 'navy'
                }
    df_final.plot(
        kind='bar',
        ax=ax,
        width=barwidth,
        color=colors
    )
    # Add annotations for the aso and hrrrspires columns
    for jdx, col in enumerate(df_final.columns):
        if 'aso' in col:
            # If this is an aso column for NWM, only label rows where the index < 2023
            # This removes the weird '0' nwm_aso labels where aso exists, but NWM does not
            if 'nwm' in col:
                labels = []
                for idx, dt in enumerate(df_final[col].index):
                    v = df_final[col].iloc[idx]
                    if dt.year < 2023 and v != 0:
                        label = f'{v*100:.0f}%'
                    else:
                        label = ''
                    labels.append(label)
                ax.bar_label(ax.containers[jdx],
                    labels=labels,
                    label_type='edge', fontsize=10, padding=barpadding, rotation=90)
            else:
                # Annotate the values on the bars if the container exists
                ax.bar_label(ax.containers[jdx],
                            labels=[f'{v*100:.0f}%' for v in df_final[col] if not np.isnan(v) and v != 0],
                            label_type='edge', fontsize=10, padding=barpadding, rotation=90)
        elif col == f'{var}_hrrrspires':
            # Annotate the values on the bars with f-str formatting, multiplying by 100 to get percentage and rotate labels by 90 degrees
            ax.bar_label(ax.containers[jdx],
                        labels=[f'{v*100:.0f}%' for v in df_final[col] if not np.isnan(v) and v != 0],
                        label_type='edge', fontsize=10, padding=barpadding, rotation=90)
        elif col == f'{var}_ua':
            # Annotate the values on the bars with f-str formatting, multiplying by 100 to get percentage and rotate labels by 90 degrees
            ax.bar_label(ax.containers[jdx],
                        labels=[f'{v*100:.0f}%' for v in df_final[col] if not np.isnan(v) and v != 0],
                        label_type='edge', fontsize=10, padding=barpadding, rotation=90)
        elif col == f'{var}_nwm':
            # Annotate the values on the bars with f-str formatting, multiplying by 100 to get percentage and rotate labels by 90 degrees
            ax.bar_label(ax.containers[jdx], color='w',
                        labels=[f'{v*100:.0f}%' for v in df_final[col] if not np.isnan(v) and v != 0],
                        label_type='edge', fontsize=10, padding=barpadding, rotation=90)

    # Abbreviate the xtick labels to %Y-%m-%d
    ax.set_xticklabels(df_final.index.strftime('%Y-%m-%d'), rotation=0)
    ax.grid('lightgrey', linestyle='--', linewidth=0.5)
    ax.set_xlabel('')
    if ylims is not None:
        ax.set_ylim(ylims)

    # Anchor legend outside of plot
    plt.legend(bbox_to_anchor=(1.025, 1), loc='upper left', fontsize=12)

    # Annotate the bar groups with the basin names
    for i, basin in enumerate(df_final['basin']):
        if basin == df_final['basin'][i-1]:
            # If the basin is the same as the previous one, skip the annotation
            continue
        else:
            # Otherwise, annotate the basin name at the bottom of the bar group
            ax.text(i, basiny, basin, ha='center', va='bottom', fontsize=12, fontweight='bold')
            # Add a vertical line to separate the basins
            if i > 0:
                ax.axvline(x=i-0.5, color='black', linestyle='-.', linewidth=1.5)

    # Change the y-axis tick labels to percentage
    ax.set_yticklabels([f'{int(v*100)}%' for v in ax.get_yticks()], fontsize=12)

plot_it_up(df_final)

In [ ]:
# Early season is before May (<5)
early_df = df_final[df_final.index.month < 5].copy()
plot_it_up(early_df, barwidth=0.6, figsize=(10, 3))

In [ ]:
# Late season is after April (>4), including May
late_df = df_final[df_final.index.month > 4].copy()
plot_it_up(late_df, barwidth=0.6, figsize=(10, 3))

In [ ]:
# Calculate the differences from the corresponding ASO reference
df_diff = df_final.copy()
for runtype in ['hrrrspires', 'nwm', 'ua']:
    # Create a new column for the difference from ASO
    df_diff[f'SCE_{runtype}_diff'] = df_diff[f'SCE_{runtype}'] - df_diff[f'SCE_{runtype}_aso']
    # Drop the corresponding aso and hrrrspires columns
    df_diff.drop(columns=[f'SCE_{runtype}_aso', f'SCE_{runtype}'], inplace=True)

df_diff

In [ ]:
# And plot it up
var = 'SCE'
colors = {
            f'{var}_nwm_diff': 'purple',
            f'{var}_hrrrspires_diff': 'dodgerblue',
            f'{var}_ua_diff': 'gold',
            f'{var}_baseline_diff': 'navy'
        }
plot_it_up(df_final=df_diff, var='SCE_diff', colors=colors)

In [ ]:
# Compute melt fractions using the valid pixel masks from above and the depth products and store the results in a dictionary for easy access
melt_results = {}
basind = ''
for a, p, m, f in zip(aso_grid, depth_products, aso_valid_pixels, product_diff_fns):
    basin = PurePath(f).stem.split('_')[0]
    dt_var = PurePath(f).stem.split('diff_')[1].split('_original')[0]
    runtype = PurePath(f).stem.split('_diff')[0].split('_')[-1]
    if basind != basin:
        print(basin)
    basind = basin

    # Compute early melt
    # Early melt is where ASO > 0 and the product is 0
    early_melt_product = p.where(a['aso_depth'] > 0)
    early_melt_product = early_melt_product.where(early_melt_product == 0)

    # Compute delayed melt
    # Sum pixels with valid data > 0 where ASO == 0
    late_melt_product = p.where(a['aso_depth'] == 0)
    late_melt_product = late_melt_product.where(late_melt_product > 0)

    # Count up the valid pixels
    early_melt_product_sum = early_melt_product.count().values
    late_melt_product_sum = late_melt_product.count().values

    # Divide by total valid pixels to get the product's melt fractions
    total_valid_pixels = m.sum().values
    early_melt_fraction = early_melt_product_sum / total_valid_pixels
    late_melt_fraction = late_melt_product_sum / total_valid_pixels
    # print(f'{dt_var} {runtype} | Early melt: {early_melt_fraction:.3f}, Late melt: {late_melt_fraction:.3f}')

    melt_results[f'{basin}_{dt_var}_{runtype}_early_melt'] = early_melt_fraction
    melt_results[f'{basin}_{dt_var}_{runtype}_delayed_melt'] = late_melt_fraction

In [ ]:
# Convert the dict into a dataframe for plotting
melt_df = pd.DataFrame(data=melt_results, index=[0]).T.rename(columns={0: 'melt_fraction'})
# Add a column for the basin and the dt and the run types
melt_df['basin'] = melt_df.index.str.split('_').str[0]
melt_df['dt'] = melt_df.index.str.split('_').str[1]
melt_df['runtype'] = melt_df.index.str.split('_').str[3]
melt_df['melttype'] = melt_df.index.str.split('_').str[4]

# And then reset the index
melt_df.reset_index(inplace=True)

# And change the 'index' column name to filename, or drop it
melt_df.rename(columns={'index': 'filename'}, inplace=True)

# Coerce dt to pandas datetime
melt_df['dt'] = pd.to_datetime(melt_df['dt'], format='%Y%m%d')

if drop_baseline:
    melt_df = melt_df[melt_df['runtype'] != 'baseline'].copy()
melt_df

In [ ]:
# Create a separate df for each of the runtypes
sub_melt_df_list = []
for r in np.unique(melt_df['runtype']):
    sub_df = melt_df[melt_df['runtype'] == r].copy()
    # Append the runtype to the SCE column name
    sub_df.rename(columns={'melt_fraction': f'melt_fraction_{r}'}, inplace=True)
    # And append the runtype to the basin column name to avoid issues with concatentation
    sub_df.rename(columns={'basin': f'basin_{r}'}, inplace=True)
    # Reset the index
    sub_df.reset_index(drop=True, inplace=True)
    # Drop the filename column, drop the runtype column
    sub_df.drop(columns=['filename', 'runtype'], inplace=True)
    sub_melt_df_list.append(sub_df)
sub_melt_df_list[1]

In [ ]:
# Combine the sub_dfs together based on the dt column
melt_df_final = pd.concat(sub_melt_df_list, axis=1)
# Create a new basin column and drop all basin_ columns
melt_df_final['basin'] = melt_df_final['basin_hrrrspires']
# Drop the basin_ columns
melt_df_final.drop(columns=[col for col in melt_df_final.columns if col.startswith('basin_')], inplace=True)
# Remove all but the first dt column encountered
melt_df_final = melt_df_final.loc[:, ~melt_df_final.columns.duplicated()]
# Set the dt column as the index
melt_df_final.set_index('dt', inplace=True)
# Clean up the column names, set the dt column as the index
melt_df_final

In [ ]:
# Split into early and late melt
early_melt_df = melt_df_final[melt_df_final['melttype'] == 'early'].copy()
late_melt_df = melt_df_final[melt_df_final['melttype'] == 'delayed'].copy()

# Drop the melttype column
early_melt_df.drop(columns=['melttype'], inplace=True)
late_melt_df.drop(columns=['melttype'], inplace=True)


In [ ]:
# and plot it up
plot_it_up(early_melt_df, var='melt_fraction', basiny=-0.15, figsize=(14, 3), barwidth=0.75, barpadding=5, ylims=(0, 0.6))
plt.title('Early Melt Fraction', x=0.075);

In [ ]:
plot_it_up(late_melt_df, var='melt_fraction', basiny=-0.15, figsize=(14, 3), barwidth=0.75, barpadding=5, ylims=(0, 0.6))
plt.title('Delayed Melt Fraction', x=0.075);

In [ ]:
# Split into early season
plot_it_up(early_melt_df[early_melt_df.index.month < 5], var='melt_fraction', basiny=-0.2, figsize=(7, 3), barwidth=0.75, barpadding=5, ylims=(0, 1))
plt.title('Early season \nearly melt fraction', ha='left', x=0.0);

# and late season
plot_it_up(early_melt_df[early_melt_df.index.month > 4], var='melt_fraction', basiny=-0.2, figsize=(7, 3), barwidth=0.75, barpadding=5, ylims=(0, 1))
plt.title('Late season \nearly melt fraction', ha='left', x=0.0);

In [ ]:
# Split into early season
plot_it_up(late_melt_df[late_melt_df.index.month < 5], var='melt_fraction', basiny=-0.2, figsize=(7, 3), barwidth=0.75, barpadding=5, ylims=(0, 1))
plt.title('Early season \ndelayed melt fraction', ha='left', x=0.0);

# and late season
plot_it_up(late_melt_df[late_melt_df.index.month > 4], var='melt_fraction', basiny=-0.2, figsize=(7, 3), barwidth=0.75, barpadding=5, ylims=(0, 1))
plt.title('Late season \ndelayed melt fraction', ha='left', x=0.0);